# Step 2: Transcribe Audio with Qwen3-ASR

This notebook uploads WAV files to Google Cloud Storage, sends them to Alibaba Cloud's Qwen3-ASR for transcription, and saves the results as JSON.

**What it does:**
1. Uploads WAV files from Google Drive to a temporary GCS bucket
2. Submits each file to Qwen3-ASR (file transcription API)
3. Polls for completion and downloads the transcript JSON
4. Cleans up the temporary GCS bucket when done

**Input:** WAV files from Step 1  
**Output:** JSON transcripts with full text and sentence-level timestamps

---
## Prerequisites
- WAV files from `01_extract_audio.ipynb`
- **Google Cloud Platform** account with Storage API enabled
- **Alibaba Cloud (DashScope)** API key — [Get one here](https://www.alibabacloud.com/en/solutions/generative-ai/dashscope)

> **Tip:** The GCS bucket is used as a temporary staging area so Qwen3-ASR can access the files via public URL. The cleanup cell at the end deletes the bucket entirely.


## Configuration

In [ ]:
import os
import time
import json
import requests
from google.colab import drive, auth
from google.cloud import storage

# Mount Google Drive
drive.mount('/content/drive')

# ===============================================
# CONFIGURATION - Edit these values
# ===============================================
DASHSCOPE_API_KEY = ""          # Your Alibaba Cloud / DashScope API key
WAV_FOLDER = "/content/drive/MyDrive/your_audio_wavs/Day_1"
OUTPUT_DIR = "/content/drive/MyDrive/your_transcripts/Day_1"
LANGUAGE = "ar"                  # Transcription language (e.g., "ar", "en", "zh")

# Google Cloud Storage (temporary staging bucket)
GCS_PROJECT_ID = "your-gcp-project-id"
GCS_BUCKET_NAME = "your-unique-bucket-name"  # Must be globally unique across GCS
GCS_BUCKET_LOCATION = "US"                    # GCS region for the temporary bucket
# ===============================================

assert DASHSCOPE_API_KEY, "Please set your DashScope API key above."
assert GCS_PROJECT_ID != "your-gcp-project-id", "Please set your GCP Project ID above."

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")


## Set Up GCS Bucket

Authenticates with GCP and creates a temporary public bucket for staging audio files.  
If the bucket already exists, it will be reused.


In [ ]:
# Authenticate with GCP (opens a browser tab for OAuth)
auth.authenticate_user()
storage_client = storage.Client(project=GCS_PROJECT_ID)

# Create or reuse the staging bucket
try:
    bucket = storage_client.get_bucket(GCS_BUCKET_NAME)
    print(f"Using existing GCS bucket: {GCS_BUCKET_NAME}")
except Exception:
    print(f"Creating new GCS bucket: {GCS_BUCKET_NAME}...")
    bucket = storage_client.create_bucket(GCS_BUCKET_NAME, location=GCS_BUCKET_LOCATION)
    bucket.uniform_bucket_level_access = True
    bucket.patch()
    print(f"Bucket created in {GCS_BUCKET_LOCATION}.")


## Upload Audio to GCS

Uploads each WAV file and makes it publicly accessible so the ASR API can fetch it.  
Files already in the bucket are skipped.


In [ ]:
wav_files = [f for f in os.listdir(WAV_FOLDER) if f.endswith('.wav')]
print(f"Found {len(wav_files)} WAV file(s).\n")

public_urls = []
for filename in wav_files:
    local_path = os.path.join(WAV_FOLDER, filename)
    blob = bucket.blob(filename)

    if blob.exists():
        url = f"https://storage.googleapis.com/{bucket.name}/{filename}"
        print(f"[exists] {filename}")
    else:
        print(f"[uploading] {filename}...")
        blob.upload_from_filename(local_path, content_type="audio/wav")
        blob.make_public()
        url = blob.public_url
        print(f"[done] {filename}")

    public_urls.append(url)

print(f"\n{len(public_urls)} file(s) ready for transcription.")


## Run Transcription

Submits each file to Qwen3-ASR one at a time and polls until complete.  
Results are saved as JSON files with full transcript text and sentence-level timestamps.

> **Note:** Each file is processed sequentially to stay within API rate limits.  
> Polling interval is 20 seconds. Long audio files may take several minutes.


In [ ]:
SUBMIT_URL = "https://dashscope-intl.aliyuncs.com/api/v1/services/audio/asr/transcription"

headers = {
    "Authorization": f"Bearer {DASHSCOPE_API_KEY}",
    "Content-Type": "application/json",
    "X-DashScope-Async": "enable",
}

for i, public_url in enumerate(public_urls):
    filename = wav_files[i]
    print(f"\n[{i+1}/{len(wav_files)}] Submitting {filename}...")

    payload = {
        "model": "qwen3-asr-flash-filetrans",
        "input": {"file_url": public_url},
        "parameters": {"language": LANGUAGE, "enable_words": False}
    }

    resp = requests.post(SUBMIT_URL, headers=headers, json=payload).json()
    if "output" not in resp:
        print(f"  [error] Submission failed: {resp}")
        continue

    task_id = resp['output']['task_id']
    print(f"  Task ID: {task_id}")

    # Poll for completion
    while True:
        status_resp = requests.get(
            f"https://dashscope-intl.aliyuncs.com/api/v1/tasks/{task_id}",
            headers={"Authorization": f"Bearer {DASHSCOPE_API_KEY}"}
        ).json()
        status = status_resp['output']['task_status']

        if status == 'SUCCEEDED':
            transcript_url = status_resp['output']['result']['transcription_url']
            transcript_data = requests.get(transcript_url).json()

            output_filename = filename.replace('.wav', '.json')
            output_path = os.path.join(OUTPUT_DIR, output_filename)
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(transcript_data, f, ensure_ascii=False, indent=4)
            print(f"  [saved] {output_filename}")
            break

        elif status in ('FAILED', 'CANCELED'):
            msg = status_resp['output'].get('message', 'Unknown error')
            print(f"  [failed] {status}: {msg}")
            break

        else:
            print(f"  [waiting] {status}...", end="\r")
            time.sleep(20)

print("\nTranscription complete.")


## Cleanup - Delete GCS Bucket

Removes the temporary staging bucket and all uploaded files.  
**Run this when you're done** to avoid GCS storage charges.


In [ ]:
try:
    bucket.delete(force=True)
    print(f"GCS bucket '{GCS_BUCKET_NAME}' deleted.")
except Exception as e:
    print(f"Error deleting bucket: {e}")
